# Visualize the Hessian

In [ ]:
import sys
sys.path.append("..")

import cv2
import torch
import numpy as np
from omegaconf import OmegaConf
import matplotlib.pyplot as plt
from curvlinops import GGNLinearOperator, HessianLinearOperator, HutchinsonSquaredFrobeniusNormEstimator


from src.models.vit_classification import VisionTransformer
from src.datasets import get_dataloader

In [ ]:
config_path = "/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/results/vit_classification/mnist/config.yaml"
conf = OmegaConf.load(config_path)
train_dataloader = get_dataloader(conf.data, train=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = VisionTransformer(**conf.model.params).to(device)

In [ ]:
model.load_state_dict(torch.load("/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/results/vit_classification/mnist/checkpoints/step_20000/model.pt", map_location="cpu", weights_only=True))

In [ ]:
params_order = [
    'embeddings.patch_embeddings.in_layer.weight',
    # 'embeddings.patch_embeddings.in_layer.bias',
    'transformer_blocks.0.self_attention.to_query.weight',
    # 'transformer_blocks.0.self_attention.to_query.bias',
    'transformer_blocks.0.self_attention.to_key.weight',
    # 'transformer_blocks.0.self_attention.to_key.bias',
    'transformer_blocks.0.self_attention.to_value.weight',
    # 'transformer_blocks.0.self_attention.to_value.bias',
    'transformer_blocks.0.self_attention.out_layer.weight',
    # 'transformer_blocks.0.self_attention.out_layer.bias',
    'transformer_blocks.0.self_attention_norm.weight',
    'transformer_blocks.0.self_attention_norm.bias',
    'transformer_blocks.0.feed_forward.in_layer.weight',
    'transformer_blocks.0.feed_forward.out_layer.weight',
    'transformer_blocks.0.feed_forward_norm.weight',
    'transformer_blocks.0.feed_forward_norm.bias',
    'out_layer.weight',
    # 'out_layer.bias'
]

attention_params_order = [
    'transformer_blocks.0.self_attention.to_query.weight',
    # 'transformer_blocks.0.self_attention.to_query.bias',
    'transformer_blocks.0.self_attention.to_key.weight',
    # 'transformer_blocks.0.self_attention.to_key.bias',
    'transformer_blocks.0.self_attention.to_value.weight',
    # 'transformer_blocks.0.self_attention.to_value.bias',
]

param_dict = {n:p for (n, p) in model.named_parameters()}
params = [param_dict[n] for n in params_order if n in param_dict]
params_attention = [param_dict[n] for n in attention_params_order if n in param_dict]
num_params = sum(p.numel() for p in params)
num_params_attention = sum(p.numel() for p in params_attention)
num_params_layer_all = [p.numel() for p in params]
num_params_layer_attention = [p.numel() for p in params_attention]

In [ ]:
num_params_layer_attention

In [ ]:
num_params_layer_all

In [ ]:
loss_function = torch.nn.CrossEntropyLoss(reduction="mean").to(device)

In [ ]:
dataloader = [next(iter(train_dataloader))]

In [ ]:
hessian_linop = HessianLinearOperator(model, loss_function, params, dataloader)
hessian_linop_attention = HessianLinearOperator(model, loss_function, params_attention, dataloader)

In [ ]:
hessian_matrix = hessian_linop @ np.eye(num_params).astype(hessian_linop.dtype)
hessian_matrix_attention = hessian_linop_attention @ np.eye(num_params_attention).astype(hessian_linop_attention.dtype)

In [ ]:
matrices = [hessian_matrix, hessian_matrix_attention]
titles = [
    "Hessian",
    "Hessian (Self-Attention)",
]
num_params_layers = [num_params_layer_all, num_params_layer_attention]

rows, columns = 1, 2
img_width = 7

def plot(transform, transform_title=None):
    min_value = min(transform(mat).min() for mat in matrices)
    max_value = max(transform(mat).max() for mat in matrices)

    # fig, axes = plt.subplots(nrows=rows, ncols=columns, sharex=True, sharey=True)
    fig, axes = plt.subplots(nrows=rows, ncols=columns, figsize=(columns * img_width, rows * img_width))

    for idx, (ax, mat, title, num_params_layer) in enumerate(zip(axes.flat, matrices, titles, num_params_layers)):
        ax.set_title(title)
        img = ax.imshow(transform(mat), vmin=min_value, vmax=max_value)
        ax.axis("off")

        # layer blocks
        boundaries = [0] + np.cumsum(num_params_layer).tolist()
        for pos in boundaries:
            if pos not in [0, num_params]:
                style = {"color": "w", "lw": 0.5, "ls": "--", "alpha": 0.9}
                ax.axhline(y=pos - 1, xmin=0, xmax=num_params - 1, **style)
                ax.axvline(x=pos - 1, ymin=0, ymax=num_params - 1, **style)

        # colorbar
        last = idx == len(matrices) - 1
        if last:
            fig.colorbar(
                img, ax=axes.ravel().tolist(), label=transform_title, shrink=0.735
            )

    return fig, axes

In [ ]:
plt.rcParams["font.size"] = 14

In [ ]:
def logabs(mat, epsilon=1e-6):
    return np.log10(np.clip(np.abs(mat), a_min=epsilon, a_max=None))

plot(logabs, transform_title="Logarithmic absolute entries")
plt.savefig("/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/figs/hessian_entries.pdf", bbox_inches="tight")
plt.show()

After training

In [ ]:
def logabs(mat, epsilon=1e-6):
    return np.log10(np.clip(np.abs(mat), a_min=epsilon, a_max=None))

plot(logabs, transform_title="Logarithmic absolute entries")
plt.savefig("/home/jovyan/shares/SR008.fs2/nkiselev/sandbox/2025-Project-182/code/figs/hessian_entries_trained.pdf", bbox_inches="tight")
plt.show()

# Block histogram

In [ ]:
V_COLOR ='#7570b3'
Q_COLOR ='#1b9e77'
FONTSIZE = 18


def plot_hists(matrix1: np.array, matrix2: np.array, key: str, matrix3: np.array = None, matrix4: np.array = None):
    plt.close()
    def h(vals, color, alpha, ax, b):
        logbins = np.logspace(max(-8, np.log10(np.min(vals))),np.log10(np.max(vals)),b)
        ax.hist(vals, edgecolor='black', color=color, bins=logbins, alpha=alpha)

    if key=='classical':
        # Create subplots: 1 row, 2 columns
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5.5, 4.5), sharex=True)  # 1 row, 2 columns
    elif key=='linear':
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.3, 3), sharey=True)  # 1 row, 2 columns

    
    # Plot histogram for V block
    if matrix3 is not None:
        h(matrix3.flatten(), color=V_COLOR, alpha=0.4, ax=ax1, b=50)
    
    h(matrix1.flatten(), color=V_COLOR, alpha=1, ax=ax1, b=50)
    ax1.set_xscale('log')
    
    # Plot histogram for Q block
    if key == 'linear':
        bins=50
        bins_opaque=30
    elif key == 'classical':
        bins=30
        bins_opaque=-1
    if matrix4 is not None:
        h(matrix4.flatten(), color=Q_COLOR, alpha=0.4, ax=ax2, b=bins_opaque)
    h(matrix2.flatten(), color=Q_COLOR, alpha=1.0, ax=ax2, b=bins)
    ax2.set_xscale('log')
    
    if key == 'classical':
        ax2.set_xlabel('Absolute Entries')
        fig.supylabel('Frequency', x=0.07, fontsize=FONTSIZE)
    elif key == 'linear':
        ax1.set_ylabel('Frequency')
        fig.supxlabel('Absolute Entries', y=0.17, fontsize=FONTSIZE)
    
    plt.tight_layout()
    # plt.savefig(f'./figures/histogram_{key}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# Query histogram
q_hessian_linop = HessianLinearOperator(model, loss_function, [param_dict['transformer_blocks.0.self_attention.to_query.weight']], dataloader)
q_hessian_mat = q_hessian_linop @ np.eye(param_dict['transformer_blocks.0.self_attention.to_query.weight'].numel()).astype(q_hessian_linop.dtype)

# Value histogram
v_hessian_linop = HessianLinearOperator(model, loss_function, [param_dict['transformer_blocks.0.self_attention.to_value.weight']], dataloader)
v_hessian_mat = v_hessian_linop @ np.eye(param_dict['transformer_blocks.0.self_attention.to_value.weight'].numel()).astype(v_hessian_linop.dtype)

In [ ]:
plot_hists(np.abs(v_hessian_mat), np.abs(q_hessian_mat), 'classical')

# Growth Rates in Self-Attention Hessian

In [ ]:
def generate_and_save_growth_rates(
    scale: str,
    norm: str,
    batch_size: int,
    num_batch: int,
    seed: int,
    n_digits: int,
    n_ctx: int,
    d_model: int,
    d_mlp: int,
    repeats: int,
    hvp: int,
    residual_scaling: float,
    num_layers_iter: Sequence,
    linear_attention: bool,
    loss_name: str,
):
    if scale == 'log':
        sigma = 10 ** np.linspace(-2, 1.0, 20)
    elif scale == 'log_short':
        sigma = 10 ** np.linspace(-1, 0.0, 20)
    elif scale == 'linear':
        sigma = np.linspace(0.1, 10.0, 20)
    elif scale == 'linear_short':
        sigma = np.linspace(0.1, 1.0, 20)
    else:
        raise ValueError('Scale not known')
    if loss_name == 'mse':
        loss_module =  MSELossModule()
    elif loss_name == 'ce':
        loss_module =  LossModule()
    else:
        raise ValueError('Unknown loss name')
    
    ds = data_generator(batch_size, n_digits, seed)
    dataset_sample = []
    for _ in range(num_batch):
       tokens = next(ds)[0]
       dataset_sample.append((tokens, tokens))
    
    
    for n_layers in num_layers_iter:
    
        frob_q = {l:{i:[] for i in range(repeats)} for l in range(n_layers)}
        frob_v = {l:{i:[] for i in range(repeats)} for l in range(n_layers)}
        frob_q_outer = {l:{i:[] for i in range(repeats)} for l in range(n_layers)}
        frob_v_outer = {l:{i:[] for i in range(repeats)} for l in range(n_layers)}
        frob_q_func = {l:{i:[] for i in range(repeats)} for l in range(n_layers)}
        frob_v_func = {l:{i:[] for i in range(repeats)} for l in range(n_layers)}
    

        for r in tqdm.tqdm(range(repeats)):
            cfg = HookedTransformerConfig(
                    n_layers = n_layers,
                    n_heads = 1,
                    d_model = d_model,
                    d_head = d_model,
                    d_mlp = d_mlp,
                    act_fn = 'relu',
                    attn_only=False,
                    normalization_type = "LN" if norm == 'pre' else None,
                    post_embedding_ln=False,
                    d_vocab=D_VOCAB,
                    d_vocab_out=D_VOCAB,
                    n_ctx=n_ctx,
                    init_weights = True,
                    device="cuda",
                    seed = seed+r,
                    residual_scaling = residual_scaling,
                    linear_attention=linear_attention,
                )
            model = HookedTransformer(cfg)
            for s in sigma:
                nn.init.normal_(model.embed.W_E, mean=0, std=s)
                nn.init.normal_(model.pos_embed.W_pos, mean=0, std=s)
        
                
                param_dict = {n:p for (n, p) in model.named_parameters()}
        
                for l in range(n_layers):
                    Hessian_linop_k = HessianLinearOperator(
                        model,
                        loss_module,
                        [param_dict[f'blocks.{l}.attn.W_Q']],
                        dataset_sample,
                        check_deterministic=False,
                    )
                    GGN_linop_k = GGNLinearOperator(
                        model,
                        loss_module,
                        [param_dict[f'blocks.{l}.attn.W_Q']],
                        dataset_sample,
                        check_deterministic=False
                    )
                    est = HutchinsonSquaredFrobeniusNormEstimator(Hessian_linop_k)
                    frob_q[l][r].append(np.sqrt(np.mean([est.sample() for _ in range(hvp)])))
                    est = HutchinsonSquaredFrobeniusNormEstimator(GGN_linop_k)
                    frob_q_outer[l][r].append(np.sqrt(np.mean([est.sample() for _ in range(hvp)])))
                    est = HutchinsonSquaredFrobeniusNormEstimator(GGN_linop_k - Hessian_linop_k)
                    frob_q_func[l][r].append(np.sqrt(np.mean([est.sample() for _ in range(hvp)])))
                    
                    Hessian_linop_v = HessianLinearOperator(
                        model,
                        loss_module,
                        [param_dict[f'blocks.{l}.attn.W_V']],
                        dataset_sample,
                        check_deterministic=False
                    )
                    GGN_linop_v = GGNLinearOperator(
                        model,
                        loss_module,
                        [param_dict[f'blocks.{l}.attn.W_V']],
                        dataset_sample,
                        check_deterministic=False
                    )
                    est = HutchinsonSquaredFrobeniusNormEstimator(Hessian_linop_v)
                    frob_v[l][r].append(np.sqrt(np.mean([est.sample() for _ in range(hvp)])))
                    est = HutchinsonSquaredFrobeniusNormEstimator(GGN_linop_v)
                    frob_v_outer[l][r].append(np.sqrt(np.mean([est.sample() for _ in range(hvp)])))
                    est = HutchinsonSquaredFrobeniusNormEstimator(GGN_linop_v - Hessian_linop_v)
                    frob_v_func[l][r].append(np.sqrt(np.mean([est.sample() for _ in range(hvp)])))
        
        
        for d, name in zip(
            [frob_v_outer, frob_v_func, frob_v, frob_q_outer, frob_q_func, frob_q],
            ['frob_v_outer', 'frob_v_func', 'frob_v', 'frob_q_outer', 'frob_q_func', 'frob_q']
        ):
          f_name = f'numerical_results/norm={norm}_num_layers={n_layers}_scale={scale}_residual_scaling={residual_scaling}_linear_attention={linear_attention}_loss={loss_name}_{name}.pickle'
          with open(f_name, 'wb') as handle:
            pickle.dump(d, handle, protocol=pickle.HIGHEST_PROTOCOL)